In [ ]:
import os

import numpy as np
import pandas as pd

import shutil

import plotly.express as px
import seaborn as sns
import matplotlib
from matplotlib import pyplot as plt

from kneebow.rotor import Rotor
from tqdm.notebook import tqdm, trange

from filter_genomes_for_download_ncbi import *

plt.rcParams["figure.dpi"] = 200
sns.set_palette("deep")
sns.set_context("paper")
sns.set_style("whitegrid")

In [ ]:
ncbi_metadata = pd.read_csv('../../data/ncbi_enterobacter_complete_metadata.tsv', sep='\t')

In [ ]:
def filter_by_species(summary, SPECIES_NAME):
    species_summary = summary[summary["genome_name"].str.contains(SPECIES_NAME)] # Filter for only SPECIES_NAME strains 
    species_summary = species_summary.dropna(subset=['genome_length']) # must have reported genome_length
    species_summary = species_summary.dropna(subset=['patric_cds']) # must have reported patric_cds
    
    # Ensure genome_length and patric_cds are ints
    species_summary['genome_length'] = species_summary.genome_length.astype('int')
    species_summary['patric_cds'] = species_summary.patric_cds.astype('int')

    return species_summary


In [ ]:
summary = pd.read_csv('../../data/genome_summary_Oct_12_23', sep='\t', dtype='object')
metadata = pd.read_csv('../../data/genome_metadata_Oct_12_23', sep='\t', dtype='object')

SPECIES_NAME = 'Enterobacter' # Just writing the genus name also works

# How many strains of the species/genus are available
species_summary = filter_by_species(summary, SPECIES_NAME)

species_metadata = metadata.loc[species_summary.index]

In [ ]:
inds_of_interest = []
accounted_for_assemblies = set()

for i, row in ncbi_metadata.iterrows():
    if row['Assembly Accession'] not in species_metadata.assembly_accession.values and row['Assembly Name'] not in accounted_for_assemblies:
        inds_of_interest.append(row.name)
    else:
        accounted_for_assemblies.add(row['Assembly Name'])

filtered_ncbi_metadata = ncbi_metadata.loc[inds_of_interest].drop_duplicates('Assembly Name', keep = 'first')

In [ ]:
# ncbi_metadata['Assembly Level'].value_counts()
ncbi_metadata.drop_duplicates('Assembly Name', keep = 'first')

In [ ]:
982 - 519

In [ ]:
df_filtration = pd.DataFrame(index = ['prefiltration', 'L50/N50', 'contig_count', 'CheckM_completeness_contamination',
                                     'min_genome_length', 'max_predicted_cds', 'gc_content'],
                             columns = ['initial', 'num_filtered', 'remaining']
                            , dtype = int)

In [ ]:
df_filtration.loc['prefiltration'] = [len(filtered_ncbi_metadata), 0, len(filtered_ncbi_metadata)]

In [ ]:
# Find reference strain N50 value from NCBI Genome and multiply by 0.85
# If your species/genus has multiple reference strains, pick the smallest by genome length
# If you are still confused, just send Sidd an email

fig, ax = plt.subplots()

# 5.3 Mb is the referene N50 value for this species (https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_000240185.1/)
species_ref_n50 = 4.2e6
min_thresh_n50 = int(0.9 * species_ref_n50)

# Most Complete sequences pass this threshold
sns.histplot(filtered_ncbi_metadata['Assembly Stats Contig N50'].dropna().astype('int'), ax=ax)
plt.axvline(x=min_thresh_n50, color='#ff00ff', linestyle='--')


filtered_ncbi_metadata = filtered_ncbi_metadata[filtered_ncbi_metadata['Assembly Stats Contig N50'] > min_thresh_n50]

In [ ]:
df_filtration.loc['L50/N50'] = [df_filtration.iloc[0,2], df_filtration.iloc[0,2] - len(filtered_ncbi_metadata), len(filtered_ncbi_metadata)]

In [ ]:
df_filtration.loc['contig_count'] = [df_filtration.iloc[1,2], df_filtration.iloc[1,2] - len(filtered_ncbi_metadata), len(filtered_ncbi_metadata)]

In [ ]:
df_filtration

In [ ]:
checkM_contamination_cutoff = 2.7 # determined in 1a through bv-brc data
checkM_completeness_cutoff = 97.4 # determined in 1a through bv-brc data

filtered_ncbi_metadata = filtered_ncbi_metadata[(filtered_ncbi_metadata['CheckM completeness'] > checkM_completeness_cutoff) &
                                                (filtered_ncbi_metadata['CheckM contamination'] < checkM_contamination_cutoff)]

In [ ]:
df_filtration.loc['CheckM_completeness_contamination'] = [df_filtration.iloc[2,2], df_filtration.iloc[2,2] - len(filtered_ncbi_metadata), len(filtered_ncbi_metadata)]

In [ ]:
min_genome_length = 4e6
max_predicted_cds = 1e4

filtered_ncbi_metadata = filtered_ncbi_metadata[(filtered_ncbi_metadata['Assembly Stats Total Sequence Length'] > min_genome_length)]

df_filtration.loc['min_genome_length'] = [df_filtration.iloc[3,2], df_filtration.iloc[3,2] - len(filtered_ncbi_metadata), len(filtered_ncbi_metadata)]

filtered_ncbi_metadata = filtered_ncbi_metadata[(filtered_ncbi_metadata['Annotation Count Gene Total'] < max_predicted_cds)]

df_filtration.loc['max_predicted_cds'] = [df_filtration.iloc[4,2], df_filtration.iloc[4,2] - len(filtered_ncbi_metadata), len(filtered_ncbi_metadata)]

In [ ]:
gc_content_min = 54
gc_content_max = 57

filtered_ncbi_metadata = filtered_ncbi_metadata[(filtered_ncbi_metadata['Assembly Stats GC Percent'] > gc_content_min) &
                                                (filtered_ncbi_metadata['Assembly Stats GC Percent'] < gc_content_max)]

df_filtration.loc['gc_content'] = [df_filtration.iloc[5,2], df_filtration.iloc[5,2] - len(filtered_ncbi_metadata), len(filtered_ncbi_metadata)]

In [ ]:
df_filtration

In [ ]:
filtered_ncbi_metadata

In [ ]:
accessions = filtered_ncbi_metadata['Assembly Accession'].values

import subprocess

download_path = '../../data/raw/genomes/'

bad_genomes = []

for acc in tqdm(accessions):
    try:
        print(f"Downloading {acc}...")
        genome_dir = os.path.join(download_path, acc)
        if not os.path.exists(genome_dir):
            print("\tMaking genome directory...")
            os.mkdir(genome_dir)

        path = os.path.join(genome_dir, f"{acc}.fna")

        cmd = f"""
        esearch -db assembly -query {acc} | \
        elink -target nuccore -format docsum | \
        efetch -format fasta > "{path}"
        """

        if os.path.isfile(path):
            None
        else:
            # Run the command in a subprocess
            subprocess.run(cmd, shell=True, check=True, executable='/bin/bash')
        
    except Exception as e:
        print(f"Error downloading {acc}: {e}")
        bad_genomes.append(acc)

In [ ]:
# some fasta files on BVBRC are empty, remove those and remove them from metadata

## command to remove files with no text in them, list made by checking file lenth in terminal
## commented out since it has been run already

count = 0
for folder in os.listdir(download_path):
    size = os.path.getsize(download_path + folder + '/' + folder + '.fna' )
    if size == 0:
        count += 1
        bad_genomes.append(folder)
        print('Removing folder:', folder)
        shutil.rmtree(os.path.join(download_path, folder))
print("Empty downloads total:", count)


In [ ]:
# remove duplicate sequences from downloaded files
from Bio import SeqIO

for folder in tqdm(os.listdir(download_path)):
    if folder in filtered_ncbi_metadata['Assembly Accession'].values:
        fpath = download_path + folder + '/' + folder + '.fna'
                
        seen = {}
        for record in SeqIO.parse(fpath, "fasta"):
            seq = str(record.seq)
            
            if seq not in seen:
                seen[seq] = record
            else:
                # prefer NZ_ version if duplicate
                if record.id.startswith("NZ_"):
                    seen[seq] = record
                elif not seen[seq].id.startswith("NZ_"):
                    pass  # keep first if neither is NZ_
        
        # overwrite the original file
        SeqIO.write(seen.values(), fpath, "fasta")
        print(f"Overwrote {folder}.fna with {len(seen)} unique sequences")

In [ ]:
len(os.listdir(download_path))

In [ ]:
filtered_ncbi_metadata = filtered_ncbi_metadata[filtered_ncbi_metadata['Assembly Accession'].apply(lambda x: x not in bad_genomes)]

# Data format processing

In [ ]:
filtered_bvbrc_summary = pd.read_csv('../../data/metadata/filtered_downloaded_species_summary.csv', index_col=0, dtype=object)
filtered_bvbrc_metadata = pd.read_csv('../../data/metadata/filtered_downloaded_species_metadata.csv', index_col=0, dtype=object)

In [ ]:
filtered_ncbi_metadata['genome_name'] = filtered_ncbi_metadata.apply(
    lambda row: f"{row['Organism Name']} {row['Organism Infraspecific Names Strain']}"
    if pd.notna(row['Organism Infraspecific Names Strain']) else row['Organism Name'],
    axis=1
)


ncbi_renamed = pd.DataFrame({
    'genome_id': filtered_ncbi_metadata['Assembly Accession'],
    'genome_name': filtered_ncbi_metadata['genome_name'],
    'taxon_id': filtered_ncbi_metadata['Organism Taxonomic ID'],
    'genome_status': filtered_ncbi_metadata['Assembly Level'],
    'genome_length': filtered_ncbi_metadata['Assembly Stats Total Sequence Length'],
    'gc_content': filtered_ncbi_metadata['Assembly Stats GC Percent'],
    'contig_l50': np.nan,  # NCBI doesn't provide L50 directly
    'contig_n50': filtered_ncbi_metadata['Assembly Stats Contig N50'],
    'chromosomes': 1,
    'plasmids': filtered_ncbi_metadata['Assembly Stats Total Number of Chromosomes'] -1 ,  # NCBI doesn't directly report plasmid count
    'contigs': filtered_ncbi_metadata['Assembly Stats Number of Contigs'],
    'patric_cds': np.nan,  # Not applicable
    'refseq_cds': filtered_ncbi_metadata['Annotation Count Gene Protein-coding'],
    'trna': np.nan,  # Not directly available
    'rrnacoarse_consistency': np.nan,  # Not available
    'fine_consistency': np.nan,        # Not available
    'checkm_completeness': filtered_ncbi_metadata['CheckM completeness'],
    'checkm_contamination': filtered_ncbi_metadata['CheckM contamination'],
    'genome_qualitydate_created': np.nan,
    'date_modified': filtered_ncbi_metadata['Assembly Release Date']  # Not available
})

# 2. Make sure column order matches that of filtered_bvbrc_summary
ncbi_renamed = ncbi_renamed[filtered_bvbrc_summary.columns]

# 3. Append to the original BVBRC dataframe
combined_summary = pd.concat([filtered_bvbrc_summary, ncbi_renamed], ignore_index=True)

combined_summary['genome_status'] = combined_summary['genome_status'].apply(lambda x: x.split()[0])

In [ ]:
import requests
import time

accessions = filtered_ncbi_metadata['Assembly BioSample Accession']

fields_of_interest = [
    "host",
    "isolation_source",
    "geo_loc_name",
    "host_disease",
    "collection_date",
    "comment"
]

results = []

headers = {
    "User-Agent": "my-python-script/1.0 (jtburrow@ucsd.edu)"  # <-- put your email
}

for acc in tqdm(accessions):
    url = f"https://api.ncbi.nlm.nih.gov/datasets/v2/biosample/accession/{acc}/biosample_report"
    
    try:
        response = requests.get(url, headers=headers, timeout=20)  # 20 sec timeout
        response.raise_for_status()  # catch HTTP errors
        data = response.json()
        
        report = data.get("reports", [])[0] if "reports" in data else None
        if report:
            attr_dict = {f: None for f in fields_of_interest}
            for attr in report.get("attributes", []):
                name = attr.get("name")
                value = attr.get("value")
                if name in attr_dict:
                    attr_dict[name] = value
            
            results.append({
                "accession": acc,
                **attr_dict
            })
    except Exception as e:
        print(f"Error fetching {acc}: {e}")
    
    # Be polite to NCBI: small delay
    time.sleep(0.4)

df = pd.DataFrame(results)

# Split geo_loc_name
if "geo_loc_name" in df.columns:
    df["country"] = df["geo_loc_name"].apply(lambda x: x.split(":")[0] if isinstance(x, str) else None)
    df["geo_full"] = df["geo_loc_name"]

In [ ]:
filtered_ncbi_and_metadata = pd.merge(filtered_ncbi_metadata, df, left_on='Assembly BioSample Accession', right_on='accession', how='left')

In [ ]:
# Construct genome_name as before
filtered_ncbi_metadata['genome_name'] = filtered_ncbi_metadata.apply(
    lambda row: f"{row['Organism Name']} {row['Organism Infraspecific Names Strain']}"
    if pd.notna(row['Organism Infraspecific Names Strain']) else row['Organism Name'],
    axis=1
)

# Create new DataFrame matching filtered_bvbrc_metadata
ncbi_metadata_formatted = pd.DataFrame({
    'genome_id': filtered_ncbi_and_metadata['Assembly Accession'],
    'genome_name': filtered_ncbi_and_metadata['genome_name'],
    'organism_name': filtered_ncbi_and_metadata['Organism Name'],
    'taxon_id': filtered_ncbi_and_metadata['Organism Taxonomic ID'],
    'genome_status': filtered_ncbi_and_metadata['Assembly Level'],
    'strain': filtered_ncbi_and_metadata['Organism Infraspecific Names Strain'],
    'serovar': np.nan,
    'biovar': np.nan,
    'pathovar': np.nan,
    'mlst': np.nan,
    'other_typing': np.nan,
    'culture_collection': np.nan,
    'type_strain': filtered_ncbi_and_metadata['Type Material Display Text'],
    'completion_date': filtered_ncbi_and_metadata['Assembly Release Date'],
    'publication': np.nan,
    'bioproject_accession': filtered_ncbi_and_metadata['Assembly BioProject Accession'],
    'biosample_accession': filtered_ncbi_and_metadata['Assembly BioSample Accession'],
    'assembly_accession': filtered_ncbi_and_metadata['Assembly Accession'],
    'genbank_accessions': filtered_ncbi_and_metadata['accession'],
    'refseq_accessions': np.nan,
    'sequencing_centers': filtered_ncbi_and_metadata['Assembly Submitter'],
    'sequencing_status': filtered_ncbi_and_metadata['ANI Check status'],
    'sequencing_platform': filtered_ncbi_and_metadata['Assembly Sequencing Tech'],
    'sequencing_depth': np.nan,
    'assembly_method': filtered_ncbi_and_metadata['Assembly Name'],
    'chromosomes': 1,
    'plasmids': filtered_ncbi_and_metadata['Assembly Stats Total Number of Chromosomes'] - 1,
    'contigs': filtered_ncbi_and_metadata['Assembly Stats Number of Contigs'],
    'sequences': filtered_ncbi_and_metadata['Assembly Stats Number of Scaffolds'],
    'genome_length': filtered_ncbi_and_metadata['Assembly Stats Total Sequence Length'],
    'gc_content': filtered_ncbi_and_metadata['Assembly Stats GC Percent'],
    'patric_cds': np.nan,
    'brc1_cds': np.nan,
    'refseq_cds': filtered_ncbi_and_metadata['Annotation Count Gene Protein-coding'],
    'isolation_site': np.nan,
    'isolation_source': filtered_ncbi_and_metadata['isolation_source'],
    'isolation_comments': np.nan,
    'collection_date': filtered_ncbi_and_metadata['collection_date'],
    'isolation_country': filtered_ncbi_and_metadata['country'],
    'geographic_location': filtered_ncbi_and_metadata['geo_loc_name'],
    'latitude': np.nan,
    'longitude': np.nan,
    'altitude': np.nan,
    'depth': np.nan,
    'other_environmental': filtered_ncbi_and_metadata['geo_full'],
    'host_name': filtered_ncbi_and_metadata['host'],
    'host_gender': np.nan,
    'host_age': np.nan,
    'host_health': filtered_ncbi_and_metadata['host_disease'],
    'body_sample_site': np.nan,
    'body_sample_subsite': np.nan,
    'other_clinical': np.nan,
    'antimicrobial_resistance': np.nan,
    'antimicrobial_resistance_evidence': np.nan,
    'gram_stain': np.nan,
    'cell_shape': np.nan,
    'motility': np.nan,
    'sporulation': np.nan,
    'temperature_range': np.nan,
    'optimal_temperature': np.nan,
    'salinity': np.nan,
    'oxygen_requirement': np.nan,
    'habitat': np.nan,
    'disease': np.nan,
    'comments': filtered_ncbi_and_metadata['comment'],
    'additional_metadata': np.nan
})

ncbi_metadata_formatted

In [ ]:
combined_metadata = pd.concat([filtered_bvbrc_metadata, ncbi_metadata_formatted], ignore_index=True)

combined_metadata['genome_status'] = combined_metadata['genome_status'].apply(lambda x: x.split()[0])

In [ ]:
combined_metadata.to_csv('../../data/metadata/filtered_downloaded_species_metadata.csv')
combined_summary.to_csv('../../data/metadata/filtered_downloaded_species_summary.csv')